# BookWise - Book Recommendation Analysis
A step-by-step walk-through of the content-based pipeline used by the Streamlit app. Run the cells top to bottom (**Kernel -> Restart & Run All**). Every number and table below is computed from your data; nothing is hard-coded.

## 1. Import libraries

In [ ]:
import sys
from pathlib import Path

# make `import src...` work when the notebook is opened from notebooks/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from src.data_preprocessing import (read_csv_flexible, preprocess_books, detect_columns,
                                    genre_counts, parse_genres)
from src.utils import DEFAULT_DATA_PATH, SAMPLE_DATA_PATH
pd.set_option("display.max_colwidth", 80)

## 2. Load dataset
Uses `data/books.csv`; falls back to the small sample file if it is missing.

In [ ]:
data_path = DEFAULT_DATA_PATH if DEFAULT_DATA_PATH.exists() else SAMPLE_DATA_PATH
print("Using:", data_path.name)
raw = read_csv_flexible(data_path)
raw.head()

## 3. Dataset inspection

In [ ]:
print("Shape:", raw.shape)
raw.info()
print("\nDetected column mapping:")
for canonical, sources in detect_columns(raw).items():
    print(f"  {canonical:13s} <- {sources}")

## 4. Missing-value analysis

In [ ]:
missing = raw.isna().sum().to_frame("missing")
missing["percent"] = (missing["missing"] / len(raw) * 100).round(2)
display(missing.sort_values("missing", ascending=False))
ax = missing["percent"].plot.barh(figsize=(7, 3.5), title="Missing values per column (%)")
ax.set_xlabel("% missing"); plt.tight_layout(); plt.show()

## 5. Duplicate analysis

In [ ]:
found = detect_columns(raw)
title_col = found["title"][0]
author_col = found["author"][0] if "author" in found else None
print("Exact duplicate rows:", int(raw.duplicated().sum()))
subset = [title_col] + ([author_col] if author_col else [])
print(f"Rows repeating the same {' + '.join(subset)}:", int(raw.duplicated(subset=subset).sum()))

## 6. Preprocessing
Cleans text, removes invalid rows, merges duplicate books and builds the features (see `src/data_preprocessing.py`).

In [ ]:
books, report = preprocess_books(raw)
print({k: v for k, v in report.items() if k not in ("columns_detected",)})
books[["title", "author", "year", "genre", "rating", "rating_count"]].head()

## 7. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
rated = books[books["rating_count"] > 0]
if len(rated):
    axes[0].hist(rated["rating"], bins=20, color="#7c3aed", edgecolor="white")
    axes[0].set_title("Distribution of average ratings"); axes[0].set_xlabel("rating")
else:
    axes[0].text(0.5, 0.5, "No rating data", ha="center"); axes[0].set_axis_off()
years = books["year"].dropna().astype(int)
if len(years):
    axes[1].hist(years, bins=30, color="#4f46e5", edgecolor="white")
    axes[1].set_title("Publication year"); axes[1].set_xlabel("year")
else:
    axes[1].text(0.5, 0.5, "No year data", ha="center"); axes[1].set_axis_off()
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
top_authors = books[books["author"] != "Unknown Author"]["author"].value_counts().head(10)
top_authors[::-1].plot.barh(ax=axes[0], color="#db2777", title="Authors with most books")
genres = genre_counts(books).head(10)
if len(genres):
    genres[::-1].plot.barh(ax=axes[1], color="#4f46e5", title="Most common genres")
else:
    axes[1].text(0.5, 0.5, "No genre data", ha="center"); axes[1].set_axis_off()
plt.tight_layout(); plt.show()

## 8. Feature engineering
Title, author, genre, publisher, description and publication decade are merged into one text field, the **soup**. Author and genre are repeated twice so they count more. Multi-word genres and publishers become single tokens (e.g. `science_fiction`).

In [ ]:
books[["title", "soup"]].head(3).to_dict("records")

## 9. TF-IDF
`TfidfVectorizer` converts each soup into a numeric vector. Words that are frequent in one book but rare overall get the highest weight.

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_features=30000,
                             sublinear_tf=True, dtype=np.float32)
tfidf = vectorizer.fit_transform(books["soup"])
print("TF-IDF matrix (books x terms):", tfidf.shape)
print("Sparsity: %.2f%% of entries are zero" % (100 * (1 - tfidf.nnz / (tfidf.shape[0] * tfidf.shape[1]))))
terms = np.array(vectorizer.get_feature_names_out())
first = tfidf[0].toarray().ravel()
print("\nHighest-weighted terms for:", books.loc[0, "title"])
for t, w in sorted(zip(terms[first > 0], first[first > 0]), key=lambda x: -x[1])[:8]:
    print(f"  {t:25s} {w:.3f}")

## 10. Cosine similarity
Cosine similarity measures the angle between two vectors: 1 = same direction, 0 = nothing in common.

In [ ]:
def similar_books(title_query, n=5):
    idx = books.index[books["title"].str.lower() == title_query.lower()]
    if len(idx) == 0:
        raise ValueError("Book not found: " + title_query)
    i = idx[0]
    sims = cosine_similarity(tfidf[i], tfidf).ravel()
    sims[i] = -1                       # exclude the book itself
    top = np.argsort(-sims)[:n]
    out = books.loc[top, ["title", "author"]].copy()
    out["similarity_%"] = (sims[top] * 100).round(1)
    return out.reset_index(drop=True)

## 11. Sample recommendations

In [ ]:
query = books.sort_values("rating_count", ascending=False)["title"].iloc[0]
print("Selected book:", query)
similar_books(query, 5)

### Same result through the production recommender (loads the saved model)

In [ ]:
from src.recommender import BookRecommender
try:
    rec = BookRecommender.load()
    display(rec.recommend_books(query, 5, use_hybrid=False)[["rank", "title", "author", "content_similarity"]])
except Exception as exc:
    print("Run `python src/train_model.py` first:", exc)

## 12. Evaluation and limitations
There is no user-interaction ground truth here, so accuracy and true Precision@K / Recall@K are **not** computed. The checks below are catalogue-based: similarity strength, diversity, coverage and metadata agreement (a *proxy*, not accuracy).

In [ ]:
from src.evaluation import evaluate_recommender
try:
    result = evaluate_recommender(rec, k=10, sample_size=100)
    display(pd.Series(result, name="value").to_frame())
except NameError:
    print("Load the recommender in the previous section first.")

In [ ]:
# Similarity-score analysis: how strong are the best matches?
sample = np.random.default_rng(42).choice(len(books), size=min(100, len(books)), replace=False)
best = []
for i in sample:
    s = cosine_similarity(tfidf[i], tfidf).ravel(); s[i] = -1
    best.append(s.max())
plt.hist(best, bins=20, color="#7c3aed", edgecolor="white")
plt.title("Similarity of the single best match (sampled books)")
plt.xlabel("cosine similarity"); plt.ylabel("books"); plt.show()
print("Mean best-match similarity: %.3f" % np.mean(best))

**Limitations**
- Quality depends on how much text the dataset has (titles only give weak signals; descriptions and genres help a lot).
- Cannot recommend by taste of a *user*, only by similarity of books (no collaborative filtering).
- Popular-book re-ranking needs rating data; otherwise only content similarity is used.
- Offline metrics are proxies; real quality needs user feedback or interaction logs.

## 13. Conclusions
- The pipeline turns raw CSV data into a reusable TF-IDF model and serves recommendations with real cosine-similarity scores.
- It is memory-aware: large catalogues use sparse matrices and on-demand similarity instead of a dense matrix.
- Next steps: collaborative filtering, user profiles, embedding/LLM-based explanations and deployment.